In [18]:
import numpy
import pysam
import matplotlib.pyplot as plt
import seaborn as sns
import pysam
import os
import pandas as pd

os.chdir("/dfs8/imarazzi_lab/share/pipelines/rna_seq/GSE253154_MTR4KO/run1")

In [19]:
meta = pd.read_csv("work/SraRunTable.csv", sep = ",")
meta["sample_id"] = meta["Sample Name"] + "_" + meta["genotype"]
meta.head(2)

,Run,age,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,cell_type,Center Name,...,Organism,Platform,ReleaseDate,create_date,version,Sample Name,source_name,SRA Study,tissue,sample_id
0,SRR27532307,P8,RNA-Seq,300,6415871400,PRJNA1064125,SAMN39420941,2223865888,KIT(+) cell,SHANGHAI INSTITUTE OF BIOCHEMISTRY AND CELL BI...,...,Mus musculus,ILLUMINA,2025-02-23T00:00:00Z,2024-01-12T21:25:00Z,1,GSM8015127,testis,SRP483429,testis,GSM8015127_Mtr4cKO
1,SRR27532309,P8,RNA-Seq,300,7804423800,PRJNA1064125,SAMN39420942,2476448813,KIT(+) cell,SHANGHAI INSTITUTE OF BIOCHEMISTRY AND CELL BI...,...,Mus musculus,ILLUMINA,2025-02-23T00:00:00Z,2024-01-12T21:26:00Z,1,GSM8015126,testis,SRP483429,testis,GSM8015126_Mtr4cKO


In [20]:
def is_extra_aa_read(read):
    if read.is_unmapped or read.is_secondary or read.is_supplementary:
        return False
    try:
        if read.get_tag("NH") != 1:
            return False
    except KeyError:
        pass
    try:
        if read.get_tag("NM") != 0:
            return False
    except KeyError:
        return False
    
    seq = read.query_sequence
    if seq is None:
        return False
    cigar = read.cigartuples
    if cigar is None:
        return False
    
    left_clip = ""
    right_clip = ""
    if cigar[0][0] == 4:
        left_clip = seq[:cigar[0][1]]
    if cigar[-1][0] == 4:
        right_clip = seq[-cigar[-1][1]:]
    
    has_aa = right_clip.endswith("AA") or left_clip.startswith("AA")
    has_tt = left_clip.startswith("TT") or right_clip.endswith("TT")
    return has_aa or has_tt

def extract_extra_aa_reads(bam_file, output_bam):
    bam = pysam.AlignmentFile(bam_file, "rb")
    out = pysam.AlignmentFile(output_bam, "wb", header=bam.header)
    
    n_total = 0
    n_extra_aa = 0
    
    for read in bam.fetch(until_eof=True):
        n_total += 1
        if n_total % 1000000 == 0:
            print(f"Processed {n_total:,} reads, {n_extra_aa:,} extraAA found...", flush=True)
        if is_extra_aa_read(read):
            out.write(read)
            n_extra_aa += 1
    
    bam.close()
    out.close()
    
    print(f"Done!")
    print(f"Total reads: {n_total:,}")
    print(f"ExtraAA reads: {n_extra_aa:,}")
    print(f"Percentage: {n_extra_aa / n_total * 100:.2f}%")
    
    sorted_bam = output_bam.replace(".bam", ".sorted.bam")
    pysam.sort("-o", sorted_bam, output_bam)
    pysam.index(sorted_bam)
    os.remove(output_bam)
    
    print(f"Saved: {sorted_bam}")
    return n_extra_aa



In [21]:
# sample = "GSM8015127_Mtr4cKO"
for sample in meta["sample_id"]:
    if sample != "GSM8015127_Mtr4cKO":

        bam_file = f"sample/combined/{sample}.filt.bam"
        output_dir = "/dfs8/imarazzi_lab/share/pipelines/rna_seq/GSE253154_MTR4KO/run1/analyses/00_polyA_reads/bams"
        # output_dir = "/dfs8/imarazzi_lab/share/jennifer/histone/analyses/gtex/05.polyA_reads/"
        # os.makedirs(output_dir, exist_ok=True)

        output = os.path.join(output_dir, f"{sample}.extraAA.bam")
        print(f"Processing {sample}...")
        extract_extra_aa_reads(bam_file, output)

Processing GSM8015126_Mtr4cKO...
Processed 1,000,000 reads, 6,713 extraAA found...
Processed 2,000,000 reads, 12,781 extraAA found...
Processed 3,000,000 reads, 18,500 extraAA found...
Processed 4,000,000 reads, 24,814 extraAA found...
Processed 5,000,000 reads, 30,967 extraAA found...
Processed 6,000,000 reads, 37,102 extraAA found...
Processed 7,000,000 reads, 43,528 extraAA found...
Processed 8,000,000 reads, 49,599 extraAA found...
Processed 9,000,000 reads, 55,726 extraAA found...
Processed 10,000,000 reads, 61,768 extraAA found...
Processed 11,000,000 reads, 67,968 extraAA found...
Processed 12,000,000 reads, 73,622 extraAA found...
Processed 13,000,000 reads, 79,950 extraAA found...
Processed 14,000,000 reads, 85,924 extraAA found...
Processed 15,000,000 reads, 91,910 extraAA found...
Processed 16,000,000 reads, 97,685 extraAA found...
Processed 17,000,000 reads, 103,741 extraAA found...
Processed 18,000,000 reads, 109,710 extraAA found...
Processed 19,000,000 reads, 115,619 ext